# Phase 7: API Smoke Test

Smoke test the FastAPI endpoints using the in-process TestClient.

In [1]:
from __future__ import annotations

from io import BytesIO
from pathlib import Path
import json
import sys

import numpy as np
from PIL import Image
from fastapi.testclient import TestClient

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.api.main import app

artifact_dir = project_root / "reports" / "artifacts"
figure_dir = project_root / "reports" / "figures"
artifact_dir.mkdir(parents=True, exist_ok=True)
figure_dir.mkdir(parents=True, exist_ok=True)

model_path = project_root / "models" / "onnx" / "efficientnet_b0_fp32.onnx"
if not model_path.exists():
    raise FileNotFoundError(f"Missing ONNX model: {model_path}")

rng = np.random.default_rng(7)
dummy = (rng.random((224, 224, 3)) * 255).astype(np.uint8)
image = Image.fromarray(dummy)
buffer = BytesIO()
image.save(buffer, format="PNG")
image_bytes = buffer.getvalue()

with TestClient(app) as client:
    health = client.get("/api/v1/health")
    ready = client.get("/api/v1/ready")
    print({"health": health.status_code, "ready": ready.status_code})
    if health.status_code != 200:
        print(health.text[:1000])
    if ready.status_code != 200:
        print(ready.text[:1000])

    predict = client.post(
        "/api/v1/predict",
        files={"file": ("sample.png", image_bytes, "image/png")},
)
    print({"predict": predict.status_code})
    if predict.status_code != 200:
        print(predict.text[:1000])
        raise RuntimeError("Predict failed")

    predict_data = predict.json()
    predict_path = artifact_dir / "api_predict_smoke.json"
    predict_path.write_text(json.dumps(predict_data, indent=2), encoding="utf-8")

predict_data

{"method": "GET", "path": "/api/v1/health", "status": 200, "duration_ms": 0.62, "event": "request_complete", "timestamp": "2026-05-29T12:33:56.240875Z", "level": "info"}


/home/selba/Desktop/ENSIAS/Mlops/PFA/code/.venv/lib/python3.10/site-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.24). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
HTTP Request: GET http://testserver/api/v1/health "HTTP/1.1 200 OK"


{"method": "GET", "path": "/api/v1/ready", "status": 200, "duration_ms": 0.72, "event": "request_complete", "timestamp": "2026-05-29T12:33:56.244031Z", "level": "info"}


HTTP Request: GET http://testserver/api/v1/ready "HTTP/1.1 200 OK"


{'health': 200, 'ready': 200}
{"method": "POST", "path": "/api/v1/predict", "status": 200, "duration_ms": 13.04, "event": "request_complete", "timestamp": "2026-05-29T12:33:56.259068Z", "level": "info"}


HTTP Request: POST http://testserver/api/v1/predict "HTTP/1.1 200 OK"


{'predict': 200}


{'predicted_class': 'benign',
 'melanoma_probability': 0.015977682545781136,
 'all_probabilities': [{'label': 'benign', 'probability': 0.984022319316864},
  {'label': 'melanoma', 'probability': 0.015977682545781136}],
 'uncertainty': 0.03195536509156227,
 'confidence_level': 'high',
 'requires_review': False,
 'review_reason': None,
 'is_calibrated': False,
 'model_version': 'efficientnet-b0-fp32-onnx',
 'latency_ms': 11.13,
 'image_hash': 'c12642582e4930fb'}

In [2]:
import base64

explain = client.post(
    "/api/v1/explain",
    files={"file": ("sample.png", image_bytes, "image/png")},
)

explain_data = explain.json()
explain_path = artifact_dir / "api_explain_smoke.json"
explain_path.write_text(json.dumps(explain_data, indent=2), encoding="utf-8")

heatmap_b64 = explain_data.get("gradcam_heatmap_b64", "")
if heatmap_b64:
    heatmap_bytes = base64.b64decode(heatmap_b64)
    heatmap_path = figure_dir / "api_explain_sample.png"
    heatmap_path.write_bytes(heatmap_bytes)
    print(f"Saved explain heatmap to {heatmap_path}")

print({"explain": explain.status_code})
explain_data.get("prediction", {})

{"method": "POST", "path": "/api/v1/explain", "status": 200, "duration_ms": 999.22, "event": "request_complete", "timestamp": "2026-05-29T12:33:59.717289Z", "level": "info"}


HTTP Request: POST http://testserver/api/v1/explain "HTTP/1.1 200 OK"


Saved explain heatmap to /home/selba/Desktop/ENSIAS/Mlops/PFA/code/reports/figures/api_explain_sample.png
{'explain': 200}


{'predicted_class': 'benign',
 'melanoma_probability': 0.015886295586824417,
 'all_probabilities': [{'label': 'benign', 'probability': 0.9841136932373047},
  {'label': 'melanoma', 'probability': 0.015886295586824417}],
 'uncertainty': 0.031772591173648834,
 'confidence_level': 'high',
 'requires_review': False,
 'review_reason': None,
 'is_calibrated': False,
 'model_version': 'efficientnet-b0-fp32-onnx',
 'latency_ms': 995.4,
 'image_hash': 'c12642582e4930fb'}